In [18]:
import os
from dotenv import load_dotenv
from pyspark.sql.window import Window
from pyspark.sql import functions as F

load_dotenv("/Users/azeemkhalipha/mlops-retail-platform/.env")

# Set Java home so PySpark can find it
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
os.environ["PATH"] = "/opt/homebrew/opt/openjdk@17/bin:" + os.environ["PATH"]

PROJECT_ROOT  = os.getenv("PROJECT_ROOT")
RAW_DATA_PATH = f"{PROJECT_ROOT}/data/raw/online_retail_II.csv"
FEATURES_PATH = f"{PROJECT_ROOT}/data/features"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data exists:  {os.path.exists(RAW_DATA_PATH)}")

Project root: /Users/azeemkhalipha/mlops-retail-platform
Data exists:  True


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("RetailFeatureEngineering") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark version: {spark.version}")
print("Spark session ready")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/07 18:54:16 WARN Utils: Your hostname, Azeems-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.3 instead (on interface en0)
26/06/07 18:54:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/07 18:54:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1
Spark session ready


Load and explore the data

In [3]:
from pyspark.sql import functions as F

# Read CSV — inferSchema automatically detects column types
df = spark.read.csv(
    RAW_DATA_PATH,
    header=True,
    inferSchema=True
)

# Rename "Customer ID" (has a space) to customer_id
df = df.withColumnRenamed("Customer ID", "customer_id")

# Show schema — what columns and types we have
df.printSchema()
print(f"\nTotal rows: {df.count():,}")
print(f"Total columns: {len(df.columns)}")
df.show(5, truncate=False)

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- customer_id: double (nullable = true)
 |-- Country: string (nullable = true)


Total rows: 1,067,371
Total columns: 8
+-------+---------+-----------------------------------+--------+-------------------+-----+-----------+--------------+
|Invoice|StockCode|Description                        |Quantity|InvoiceDate        |Price|customer_id|Country       |
+-------+---------+-----------------------------------+--------+-------------------+-----+-----------+--------------+
|489434 |85048    |15CM CHRISTMAS GLASS BALL 20 LIGHTS|12      |2009-12-01 07:45:00|6.95 |13085.0    |United Kingdom|
|489434 |79323P   |PINK CHERRY LIGHTS                 |12      |2009-12-01 07:45:00|6.75 |13085.0    |United Kingdom|
|489434 |79323W   | WHITE CHERRY LI

In [4]:
# Invoices starting with 'C' are cancellations (e.g., C536379)
# These have negative quantities and should not be in training data
before = df.count()
df = df.filter(~F.col("Invoice").startswith("C"))
print(f"Removed cancellations: {before - df.count():,} rows")

Removed cancellations: 19,494 rows


In [ ]:
# Remove rows where quantity or price is zero or negative
# A sale must have positive quantity and price
df = df.filter(F.col("Quantity") > 0)
df = df.filter(F.col("Price") > 0)
print(f"After quality filter: {df.count():,} rows")

After quality filter: 1,041,670 rows


In [6]:
# Remove rows where customer_id is null
# We need customer_id for customer-level features
df = df.dropna(subset=["customer_id"])
print(f"After dropping null customers: {df.count():,} rows")

After dropping null customers: 805,549 rows


In [10]:
# Convert InvoiceDate string to actual date type
df = df.withColumn(
    "invoice_date",
    F.to_date(F.col("InvoiceDate"))
)

# Extract useful date parts
df = df.withColumn("year",        F.year("invoice_date"))
df = df.withColumn("month",       F.month("invoice_date"))
df = df.withColumn("day",         F.dayofmonth("invoice_date"))
df = df.withColumn("dayofweek",   F.dayofweek("invoice_date"))
df = df.withColumn("quarter",     F.quarter("invoice_date"))
df = df.withColumn("weekofyear",  F.weekofyear("invoice_date"))

# Flag weekends (dayofweek: 1=Sunday, 7=Saturday in Spark)
df = df.withColumn(
    "is_weekend",
    F.when(F.col("dayofweek").isin([1, 7]), 1).otherwise(0)
)

# Flag end of month (day >= 25)
df = df.withColumn(
    "is_month_end",
    F.when(F.col("day") >= 25, 1).otherwise(0)
)

# Add revenue column
df = df.withColumn(
    "revenue",
    F.round(F.col("Quantity") * F.col("Price"), 2)
)

# Verify
df.select(
    "invoice_date", "year", "month", "dayofweek",
    "is_weekend", "is_month_end", "revenue"
).show()

+------------+----+-----+---------+----------+------------+-------+
|invoice_date|year|month|dayofweek|is_weekend|is_month_end|revenue|
+------------+----+-----+---------+----------+------------+-------+
|  2009-12-01|2009|   12|        3|         0|           0|   83.4|
|  2009-12-01|2009|   12|        3|         0|           0|   81.0|
|  2009-12-01|2009|   12|        3|         0|           0|   81.0|
|  2009-12-01|2009|   12|        3|         0|           0|  100.8|
|  2009-12-01|2009|   12|        3|         0|           0|   30.0|
|  2009-12-01|2009|   12|        3|         0|           0|   39.6|
|  2009-12-01|2009|   12|        3|         0|           0|   30.0|
|  2009-12-01|2009|   12|        3|         0|           0|   59.5|
|  2009-12-01|2009|   12|        3|         0|           0|   30.6|
|  2009-12-01|2009|   12|        3|         0|           0|   45.0|
|  2009-12-01|2009|   12|        3|         0|           0|   39.6|
|  2009-12-01|2009|   12|        3|         0|  

In [13]:
df.columns

['Invoice',
 'StockCode',
 'Description',
 'Quantity',
 'InvoiceDate',
 'Price',
 'customer_id',
 'Country',
 'invoice_date',
 'year',
 'month',
 'day',
 'dayofweek',
 'quarter',
 'weekofyear',
 'is_weekend',
 'is_month_end',
 'revenue']

In [14]:
df.show()

+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+------------+----+-----+---+---------+-------+----------+----------+------------+-------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|customer_id|       Country|invoice_date|year|month|day|dayofweek|quarter|weekofyear|is_weekend|is_month_end|revenue|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+------------+----+-----+---+---------+-------+----------+----------+------------+-------+
| 489434|    85048|15CM CHRISTMAS GL...|      12|2009-12-01 07:45:00| 6.95|    13085.0|United Kingdom|  2009-12-01|2009|   12|  1|        3|      4|        49|         0|           0|   83.4|
| 489434|   79323P|  PINK CHERRY LIGHTS|      12|2009-12-01 07:45:00| 6.75|    13085.0|United Kingdom|  2009-12-01|2009|   12|  1|        3|      4|        49|         0|           0|   81.0|
| 489434|   79323W| WHITE CHERRY LIGHTS|

In [15]:
# Before building lag features, we need daily totals per product
# Right now we have one row per transaction
# We need one row per (product, date)

daily = df.groupBy("StockCode", "invoice_date").agg(
    F.sum("Quantity").alias("daily_qty"),
    F.sum("revenue").alias("daily_revenue"),
    F.countDistinct("Invoice").alias("daily_orders")
)

daily = daily.orderBy("StockCode", "invoice_date")

print(f"Daily aggregated rows: {daily.count():,}")
daily.show(10)

Daily aggregated rows: 439,167


+---------+------------+---------+------------------+------------+
|StockCode|invoice_date|daily_qty|     daily_revenue|daily_orders|
+---------+------------+---------+------------------+------------+
|    10002|  2009-12-01|       12|              10.2|           1|
|    10002|  2009-12-03|        7|              5.95|           3|
|    10002|  2009-12-04|       73|             62.05|           4|
|    10002|  2009-12-06|       49|             41.65|           2|
|    10002|  2009-12-07|        2|               1.7|           1|
|    10002|  2009-12-08|       12|              10.2|           1|
|    10002|  2009-12-11|        9|              7.65|           1|
|    10002|  2009-12-14|       36|30.599999999999998|           2|
|    10002|  2009-12-21|       12|              10.2|           1|
|    10002|  2009-12-23|        1|              0.85|           1|
+---------+------------+---------+------------------+------------+
only showing top 10 rows


In [22]:
from pyspark.sql.window import Window

# Window: look at previous rows for the SAME product, ordered by date
# This is what allows us to say "what was the quantity 7 days ago FOR THIS PRODUCT"
w = Window.partitionBy("StockCode").orderBy("invoice_date")

# Lag features — what happened N days ago
# F.lag("daily_qty", 1) means: look 1 row back in the window
daily = daily.withColumn("qty_lag_1",  F.lag("daily_qty", 1).over(w))
daily = daily.withColumn("qty_lag_7",  F.lag("daily_qty", 7).over(w))
daily = daily.withColumn("qty_lag_30", F.lag("daily_qty", 30).over(w))

# Show what lag features look like
daily.select(
    "StockCode", "invoice_date",
    "daily_qty", "qty_lag_1", "qty_lag_7", "qty_lag_30"
).show(15)

+---------+------------+---------+---------+---------+----------+
|StockCode|invoice_date|daily_qty|qty_lag_1|qty_lag_7|qty_lag_30|
+---------+------------+---------+---------+---------+----------+
|   10123C|  2009-12-04|        1|     NULL|     NULL|      NULL|
|   10123C|  2009-12-08|        2|        1|     NULL|      NULL|
|   10123C|  2009-12-10|       10|        2|     NULL|      NULL|
|   10123C|  2009-12-11|        2|       10|     NULL|      NULL|
|   10123C|  2009-12-14|       14|        2|     NULL|      NULL|
|   10123C|  2009-12-17|        1|       14|     NULL|      NULL|
|   10123C|  2009-12-18|      108|        1|     NULL|      NULL|
|   10123C|  2010-01-15|        1|      108|        1|      NULL|
|   10123C|  2010-01-20|       12|        1|        2|      NULL|
|   10123C|  2010-01-31|       12|       12|       10|      NULL|
|   10123C|  2010-02-14|        1|       12|        2|      NULL|
|   10123C|  2010-02-18|       12|        1|       14|      NULL|
|   10123C

In [23]:
# Rolling window: look at the PREVIOUS N rows (not including current)
# rowsBetween(-7, -1) means: look from 7 rows ago to 1 row ago
w7  = Window.partitionBy("StockCode") \
            .orderBy("invoice_date") \
            .rowsBetween(-7, -1)

w30 = Window.partitionBy("StockCode") \
            .orderBy("invoice_date") \
            .rowsBetween(-30, -1)

# 7-day rolling average — smooth short-term trend
daily = daily.withColumn(
    "qty_rolling_avg_7",
    F.round(F.avg("daily_qty").over(w7), 2)
)

# 30-day rolling average — smooth long-term trend
daily = daily.withColumn(
    "qty_rolling_avg_30",
    F.round(F.avg("daily_qty").over(w30), 2)
)

# 7-day rolling std — how volatile is demand?
# High std = unpredictable product, low std = stable product
daily = daily.withColumn(
    "qty_rolling_std_7",
    F.round(F.stddev("daily_qty").over(w7), 2)
)

# Drop rows where lag features are null (not enough history)
daily = daily.dropna(subset=["qty_lag_30"])

print(f"Rows after dropping nulls: {daily.count():,}")
daily.select(
    "StockCode", "invoice_date", "daily_qty",
    "qty_rolling_avg_7", "qty_rolling_avg_30", "qty_rolling_std_7"
).show(10)

Rows after dropping nulls: 330,546


+---------+------------+---------+-----------------+------------------+-----------------+
|StockCode|invoice_date|daily_qty|qty_rolling_avg_7|qty_rolling_avg_30|qty_rolling_std_7|
+---------+------------+---------+-----------------+------------------+-----------------+
|   10123C|  2010-09-27|      144|            28.43|             15.17|            51.13|
|   10123C|  2010-10-03|        1|            28.43|             19.93|            51.13|
|   10123C|  2010-10-11|        2|            28.29|              19.9|            51.21|
|   10123C|  2010-10-24|        1|            27.86|             19.63|            51.45|
|   10123C|  2010-10-25|        1|            26.29|              19.6|            52.18|
|   10123C|  2010-11-01|        2|            24.71|             19.17|            52.84|
|   10123C|  2010-11-03|       14|            23.29|              19.2|            53.38|
|   10123C|  2010-11-17|        1|            23.57|             16.07|            53.31|
|   10123C

In [24]:
# Calculate stats per customer across the whole dataset
customer_stats = df.groupBy("customer_id").agg(
    F.sum("revenue").alias("customer_total_spend"),
    F.countDistinct("Invoice").alias("customer_total_orders"),
    F.avg("revenue").alias("customer_avg_order_value"),
    F.countDistinct("StockCode").alias("customer_unique_products"),
    F.min("invoice_date").alias("customer_first_purchase"),
    F.max("invoice_date").alias("customer_last_purchase")
)

# Customer lifetime in days
customer_stats = customer_stats.withColumn(
    "customer_lifetime_days",
    F.datediff(
        F.col("customer_last_purchase"),
        F.col("customer_first_purchase")
    )
)

# Segment customers by total spend
# High value: spent more than 5000
# Mid value: spent between 1000 and 5000
# Low value: spent less than 1000
customer_stats = customer_stats.withColumn(
    "customer_segment",
    F.when(F.col("customer_total_spend") > 5000, "high_value")
     .when(F.col("customer_total_spend") > 1000, "mid_value")
     .otherwise("low_value")
)

print(f"Unique customers: {customer_stats.count():,}")
customer_stats.show(5)

Unique customers: 5,878


+-----------+--------------------+---------------------+------------------------+------------------------+-----------------------+----------------------+----------------------+----------------+
|customer_id|customer_total_spend|customer_total_orders|customer_avg_order_value|customer_unique_products|customer_first_purchase|customer_last_purchase|customer_lifetime_days|customer_segment|
+-----------+--------------------+---------------------+------------------------+------------------------+-----------------------+----------------------+----------------------+----------------+
|    17072.0|  282.04999999999995|                    1|      12.820454545454544|                      20|             2010-03-24|            2010-03-24|                     0|       low_value|
|    17884.0|  3072.8900000000003|                   20|       6.375290456431536|                     305|             2009-12-04|            2011-12-06|                   732|       mid_value|
|    16822.0|  181.39000000000

In [25]:
import os

os.makedirs(f"{FEATURES_PATH}/ml_features",       exist_ok=True)
os.makedirs(f"{FEATURES_PATH}/customer_features",  exist_ok=True)

# Save as Parquet — much faster to read than CSV, preserves data types
daily.write.mode("overwrite").parquet(
    f"{FEATURES_PATH}/ml_features"
)

customer_stats.write.mode("overwrite").parquet(
    f"{FEATURES_PATH}/customer_features"
)

print(f"ML features saved:       {daily.count():,} rows, {len(daily.columns)} columns")
print(f"Customer features saved: {customer_stats.count():,} rows")
print(f"\nML feature columns: {daily.columns}")

ML features saved:       330,546 rows, 11 columns
Customer features saved: 5,878 rows

ML feature columns: ['StockCode', 'invoice_date', 'daily_qty', 'daily_revenue', 'daily_orders', 'qty_lag_1', 'qty_lag_7', 'qty_lag_30', 'qty_rolling_avg_7', 'qty_rolling_avg_30', 'qty_rolling_std_7']


26/06/08 00:48:09 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:131)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:707)
	at org.apache.spark.storage.BlockManagerMasterE